# Sing-cell DNA Methylation Analysis with `cytozip`
Walkthrough the commonly used steps for single-cell DNA methylation analysis

In [1]:
import os,sys
import pandas as pd

download example dataset: https://figshare.com/articles/dataset/cytozip_example_data/32095567
```shell
pip install pyfigshare # https://github.com/DingWB/pyfigshare
figshare download 32095567 --cpu 4 --outdir cytozip_example_data # download the whole folder (cytozip_example_data)
```

In [2]:
os.chdir(os.path.expanduser("~/Projects/test_cytozip"))

## 1. Build the reference `.cz`

The reference holds the genome-wide `(chrom, pos, strand, context)`
axis. Per-cell `.cz` then store only `mc`/`cov` and reuse the
reference's positions, cutting per-cell size by ~5×.

In [4]:
!czip build_ref --help

usage: czip build_ref [-h] -g GENOME [-O OUTPUT] [-p PATTERN] [-j JOBS]
                      [--keep_temp] [-s CHROMS] [--no_delta]

options:
  -h, --help            show this help message and exit
  -g GENOME, --genome GENOME
                        reference genome FASTA (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: hg38_allc.cz)
  -p PATTERN, --pattern PATTERN
                        nucleotide pattern (default: C)
  -j JOBS, --jobs JOBS  number of parallel processes (CPUs) (default: 12)
  --keep_temp           keep temp directory (default: False)
  -s CHROMS, --chroms CHROMS
                        Path to a `.fai` index file or a plain text file whose
                        first (tab-separated, no header) column lists
                        chromosome names. When provided, only these
                        chromosomes are extracted, and the merged reference
                        `.cz` stores chunks in exactly this order. `Non

In [6]:
! head ~/Ref/mm10/mm10_ucsc_with_chrL.chrom.sizes
# -s / --chroms (optional): a .fai index or single-column text file of
# chromosome names. Only these chromosomes are extracted and the reference
# .cz is built in this exact order. Omit it to process every sequence.

chr1	195471971
chr10	130694993
chr11	122082543
chr12	120129022
chr13	120421639
chr14	124902244
chr15	104043685
chr16	98207768
chr17	94987271
chr18	90702639


In [5]:
!time czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O ~/Ref/mm10/mm10_with_chrL.allc.cz -j 20 \
    -s ~/Ref/mm10/mm10_ucsc_with_chrL.chrom.sizes
# or use the downloaded reference mm10 cz file (mm10_with_chrL.allc.cz) under cytozip_example_data

2026-07-26 16:32:03.441 | DEBUG    | cytozip.allc:WriteC:66 - chr1
2026-07-26 16:32:04.091 | DEBUG    | cytozip.allc:WriteC:66 - chr10
2026-07-26 16:32:04.731 | DEBUG    | cytozip.allc:WriteC:66 - chr11
2026-07-26 16:32:05.361 | DEBUG    | cytozip.allc:WriteC:66 - chr12
2026-07-26 16:32:06.025 | DEBUG    | cytozip.allc:WriteC:66 - chr13
2026-07-26 16:32:06.693 | DEBUG    | cytozip.allc:WriteC:66 - chr14
2026-07-26 16:32:07.234 | DEBUG    | cytozip.allc:WriteC:66 - chr15
2026-07-26 16:32:07.747 | DEBUG    | cytozip.allc:WriteC:66 - chr16
2026-07-26 16:32:08.249 | DEBUG    | cytozip.allc:WriteC:66 - chr17
2026-07-26 16:32:08.735 | DEBUG    | cytozip.allc:WriteC:66 - chr18
2026-07-26 16:32:09.044 | DEBUG    | cytozip.allc:WriteC:66 - chr19
2026-07-26 16:32:10.093 | DEBUG    | cytozip.allc:WriteC:66 - chr2
2026-07-26 16:32:10.927 | DEBUG    | cytozip.allc:WriteC:66 - chr3
2026-07-26 16:32:11.751 | DEBUG    | cytozip.allc:WriteC:66 - chr4
2026-07-26 16:32:12.575 | DEBUG    | cytozip.allc:Wr

In [9]:
!czip header -I cytozip_example_data/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.37
total_size  :  1322265693
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [10]:
! czip view -I cytozip_example_data/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	3000003	+	CTG
chr1	3000005	-	CAG
chr1	3000009	+	CTA
chr1	3000016	-	CAA
chr1	3000018	-	CAC
chr1	3000019	-	CCA
chr1	3000023	+	CTT
chr1	3000027	-	CAA
chr1	3000029	-	CTC


In [11]:
! czip summary -I cytozip_example_data/mm10_with_chrL.allc.cz | head

chrom	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	102	95328851	95386814	3615	78962721
chr10	95386814	63310564	158735944	2409	52609184
chr11	158735944	60770669	219544747	2382	52027265
chr12	219544747	58457323	278037836	2234	48799752
chr13	278037836	58621213	336694783	2232	48750883
chr14	336694783	60350467	397081896	2289	49987736
chr15	397081896	50425550	447538412	1934	42230765
chr16	447538412	47066788	494633718	1781	38899643
chr17	494633718	46270276	540932704	1793	39153472


## 2. Build CG / CH context index from the reference

The full reference `.cz` built above contains **every C** in the genome
(pattern `C`). Most analyses only care about specific contexts —
`CGN` (mCG / 5mC / 5hmC) or `CHN` (mCH, where H = A/C/T). Instead of
re-scanning the genome each time, cytozip builds a tiny **context
index** that records the reference row IDs of every site matching the
desired pattern. The same index can then be reused by `extractCG`,
`aggregate`, `query`, and `cz_to_anndata`.

In [12]:
!czip index --help

usage: czip index [-h] {context,regions,probes} ...

positional arguments:
  {context,regions,probes}
                        index kind: context | regions | probes
    context             Index sites by sequence context (CGN/CHN/+CGN)
    regions             Index sites by genomic regions from a BED file
    probes              Index methylation array probes (EPIC / 450K) — NOT YET
                        IMPLEMENTED

options:
  -h, --help            show this help message and exit


In [13]:
!czip index context --help

usage: czip index context [-h] -I INPUT [-O OUTPUT] [-p PATTERN] [-j JOBS]
                          [-k CHUNK_KEYS]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input reference .cz file (default: None)
  -O OUTPUT, --output OUTPUT
                        output index .cz file (default: None)
  -p PATTERN, --pattern PATTERN
                        IUPAC context pattern, optional +/- strand prefix
                        (e.g. CGN, CHN, +CGN, CAC, CAG, CHG, CWG) (default:
                        CGN)
  -j JOBS, --jobs JOBS  number of parallel processes (one shard per chunk key)
                        (default: 4)
  -k CHUNK_KEYS, --chunk-keys CHUNK_KEYS
                        restrict to chunk keys (typically chromosomes): comma-
                        separated list (chr1,chr2,...) or path to a chrom-
                        sizes-like file (uses the first whitespace-separated
                        column) (def

In [14]:
# Generate CG and CH context index for a given reference .cz file
! time czip index context -I cytozip_example_data/mm10_with_chrL.allc.cz -O cytozip_example_data/mm10_with_chrL.CGN.cz \
    --pattern CGN -j 16


real	0m22.851s
user	3m44.587s
sys	0m7.065s


In [16]:
! time czip index context -I cytozip_example_data/mm10_with_chrL.allc.cz -O cytozip_example_data/mm10_with_chrL.CHN.cz \
    --pattern CHN -j 16


real	1m4.171s
user	11m10.927s
sys	0m10.055s


mm10_with_chrL.CGN.cz and mm10_with_chrL.CHN.cz is a tiny .cz "row-pointer" file. It stores, per chromosome, the 1-based row numbers of every CpG or non-CpG cytosine in the reference mm10_with_chrL.allc.cz — just one little-endian uint32 column named ID, no positions / strands / counts.

Think of it as a bookmark list: "these are the rows in the reference that are CHN sites." Downstream commands (call_dmr_ch -s, extractCG, allc2cz --reference, aggregate --index, ...) take this file as --index, much faster than re-scanning the full reference.

In [18]:
ls cytozip_example_data/mm10_with_chrL*.cz -sh

1.3G cytozip_example_data/mm10_with_chrL.allc.cz
 31M cytozip_example_data/mm10_with_chrL.CGN.cz
 50M cytozip_example_data/mm10_with_chrL.CHN.cz


In [20]:
# index .cz is also a cz format file and can be viewed using czip too
! czip header -I cytozip_example_data/mm10_with_chrL.CGN.cz

magic  :  b'CZIP'
version  :  0.37
total_size  :  31890013
message  :  mm10_with_chrL.allc.cz
formats  :  ['I']
columns  :  ['ID']
sort_col  :  None
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  57


In [21]:
! czip view -I cytozip_example_data/mm10_with_chrL.CGN.cz | head

ID
298
299
352
353
358
359
458
459
587


In [22]:
! czip view -I cytozip_example_data/mm10_with_chrL.CGN.cz -r cytozip_example_data/mm10_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context	ID
chr1	3000827	+	CGT	298
chr1	3000828	-	CGG	299
chr1	3001007	+	CGG	352
chr1	3001008	-	CGA	353
chr1	3001018	+	CGT	358
chr1	3001019	-	CGG	359
chr1	3001277	+	CGA	458
chr1	3001278	-	CGA	459
chr1	3001629	+	CGT	587


In [23]:
# can also be queried
! czip view -I cytozip_example_data/mm10_with_chrL.CHN.cz --show_dims 0 -K chr9 | head

chrom	ID
chr9	2
chr9	3
chr9	4
chr9	5
chr9	6
chr9	7
chr9	8
chr9	9
chr9	10


Once a context index is built, you can pass it to other commands:

```shell
# Subset a per-cell .cz to CG sites only
czip extractCG -I INPUT_CZ -O OUTPUT_CGN_CZ --index cytozip_example_data/mm10_with_chrL.CGN.cz

# Or query a region in the CG-only space
czip query -I INPUT_CZ -r cytozip_example_data/mm10_with_chrL.CGN.cz -K chr1 -s 3000000 -e 3010000
```

For **region-based** indices (e.g. genes ± 2 kb from a BED file)
see [`czip index regions`](dev.ipynb) in the Advanced Features
notebook.


## 3. BAM → `.cz` & generating cz format in mapping pipeline

Convert a position-sorted BAM directly to `.cz` (no intermediate ALLC
text). The output has only `mc` / `cov` because we pass a reference.


czip bam_to_cz or the corresponding Python API can be used to call DNA methylation from a bam file and generate .cz output<br/>

### 3.1 Generating .cz format in mapping pipeline
Generation of .cz format has already been implemented in YAP pipeline.<br/>
See YAP pipeline: https://github.com/DingWB/cemba_data for detail information. <br/>
Example usage:

```shell
# step1: demultiplex
yap-gcp run_demultiplex --fq_dir=FASTQ_FOLDER --outdir="mapping" --gcp=False --n_jobs=16 --print_only=True 

# step2: Mapping
# generate mapping configure file
yap default-mapping-config --mode m3c --barcode_version V2 \
  --hisat3n_dna_ref "~/Ref/mm10/mm10_ucsc_with_chrL" \ # prefix for hisat-3n index files
  --genome "~/Ref/mm10/mm10_ucsc_with_chrL.fa" \
  --chrom_size_path "~/Ref/mm10/mm10_ucsc_with_chrL.chrom.sizes" \
  --mc_format cz --reference_cz "~/Ref/mm10/mm10_with_chrL.allc.cz" > m3c_config.ini

# Run mapping
yap-gcp run_mapping --workd="mapping" --gcp=False --config_path="m3c_config.ini" --aligner='hisat-3n' --n_jobs=64 --print_only=True
```

### 3.2 Generating .cz file from position-sorted bam files

In [41]:
! mkdir -p cytozip_example_data/mm10_cz
! ls cytozip_example_data/mm10_bam -sh

total 705M
141M FC_M_P3b_3C_6-6-J3-P24.hisat3n_dna.all_reads.deduped.bam
4.0M FC_M_P3b_3C_6-6-J3-P24.hisat3n_dna.all_reads.deduped.bam.bai
 86M FC_M_P6a_3C_7-3-K21-P5.hisat3n_dna.all_reads.deduped.bam
4.5M FC_M_P6a_3C_7-3-K21-P5.hisat3n_dna.all_reads.deduped.bam.bai
 31M FC_M_P9B_3C_6-2-F6-O4.hisat3n_dna.all_reads.deduped.bam
4.0M FC_M_P9B_3C_6-2-F6-O4.hisat3n_dna.all_reads.deduped.bam.bai
249M FC_P13a_3C_7-1-A11-O1.hisat3n_dna.all_reads.deduped.bam
3.5M FC_P13a_3C_7-1-A11-O1.hisat3n_dna.all_reads.deduped.bam.bai
178M FC_P28a_3C_2-1-E5-N14.hisat3n_dna.all_reads.deduped.bam
4.0M FC_P28a_3C_2-1-E5-N14.hisat3n_dna.all_reads.deduped.bam.bai


In [48]:
! for bam_file in `ls cytozip_example_data/mm10_bam/*.bam`; do \
        cell_id=$(echo ${bam_file/.hisat3n_dna.all_reads.deduped.bam/} | cut -d "/" -f 3); \
        echo ${cell_id}; \
        czip bam_to_cz -I ${bam_file} --genome ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O cytozip_example_data/mm10_cz/${cell_id}.cz \
        --mode mc_cov --count_fmt B --reference cytozip_example_data/mm10_with_chrL.allc.cz \
        --chroms ~/Ref/mm10/mm10_ucsc_with_chrL.chrom.sizes; \
done

FC_M_P3b_3C_6-6-J3-P24
[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/mm10_bam/FC_M_P3b_3C_6-6-J3-P24.hisat3n_dna.all_reads.deduped.bam.bai
2026-07-26 17:00:15.089 | WARNING  | cytozip.bam:_handle_site:1076 - mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
FC_M_P6a_3C_7-3-K21-P5
2026-07-26 17:01:30.245 | WARNING  | cytozip.bam:_handle_site:1076 - mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
FC_M_P9B_3C_6-2-F6-O4
2026-07-26 17:02:38.188 | WARNING  | cytozip.bam:_handle_site:1076 - mc/cov value exceeds count_fmt='B' max (255); clipping. Consider count_fmt='H' for bulk/high-coverage data.
FC_P13a_3C_7-1-A11-O1
[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/mm10_bam/FC_P13a_3C_7-1-A11-O1.hisat3n_dna.all_reads.deduped.bam.bai
2026-07-26 17:03:20.061 | WARNING  | cytozip.bam:_handle_site:1076 - mc/

## 4. `catcz` and `merge_cz`

* `catcz` — concatenate per-cell `.cz` into one multi-cell `.cz` by
  adding a `cell_id` dimension.
* `merge_cz` — sum `mc` / `cov` across cells (pooled pseudobulk).


In [20]:
! ls output/cz/*.cz -sh

 76M output/cz/FC_E17b_3C_5-5-I24-A21.cz
 22M output/cz/FC_M_E15a_3C_1-1-I5-B1.cz
 29M output/cz/FC_M_P12b_3C_2-5-M17-N10.cz
 19M output/cz/FC_M_P3b_3C_6-6-J3-P24.cz
 12M output/cz/FC_M_P6a_3C_7-3-K21-P5.cz
6.0M output/cz/FC_M_P9B_3C_6-2-F6-O4.cz
 51M output/cz/FC_P0b_3C_5-1-I24-J14.cz
 31M output/cz/FC_P13a_3C_7-1-A11-O1.cz
 23M output/cz/FC_P28a_3C_2-1-E5-N14.cz


In [21]:
! time czip catcz -O output/all_cells.cz -I "output/cz/*.cz" --key_added cell_id -F B,B -C mc,cov


real	0m1.424s
user	0m0.206s
sys	0m0.613s


In [22]:
! czip header -I output/all_cells.cz

magic  :  b'CZIP'
version  :  0.3
total_size  :  277478530
message  :  
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_dims  :  ['chrom', 'cell_id']
header_size  :  47


In [23]:
! czip summary -I output/all_cells.cz | head

chrom	cell_id	chunk_start_offset	chunk_size	chunk_tail_offset	chunk_nblocks	chunk_nrows
chr1	FC_E17b_3C_5-5-I24-A21	47	6002191	6007106	603	78962721
chr10	FC_E17b_3C_5-5-I24-A21	6007106	3999949	10010316	402	52609184
chr11	FC_E17b_3C_5-5-I24-A21	10010316	3835747	13849284	397	52027265
chr12	FC_E17b_3C_5-5-I24-A21	13849284	3618595	17470908	373	48799752
chr13	FC_E17b_3C_5-5-I24-A21	17470908	3703795	21177724	372	48750883
chr14	FC_E17b_3C_5-5-I24-A21	21177724	3678031	24858856	382	49987736
chr15	FC_E17b_3C_5-5-I24-A21	24858856	3173077	28034562	323	42230765
chr16	FC_E17b_3C_5-5-I24-A21	28034562	2939286	30976269	297	38899643
chr17	FC_E17b_3C_5-5-I24-A21	30976269	2877415	33856121	299	39153472


In [24]:
# merge 9 single-cell .cz files into a pseudobulk .cz file, summing the mc and cov values across all cells:
! time czip merge_cz -i output/cz -O output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -F H,H -j 20

2026-04-26 23:52:18.431 | INFO     | cytozip.merge:merge_cz:492 - output/pseudobulk.cz
2026-04-26 23:52:25.244 | INFO     | cytozip.merge:_bg_rmtree:141 - Removing temp dir /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/pseudobulk.cz.tmp (in background)

real	0m7.040s
user	1m32.465s
sys	0m4.578s


In [25]:
! czip view -I output/pseudobulk.cz --show_dims 0 \
    -r output/mm10_with_chrL.allc.cz | awk '$6 >10' | head

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520474	-	CAG	13	13
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [26]:
! czip query -I output/pseudobulk.cz \
    -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482

chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	11	11
chr1	25520458	-	CCA	11	11
chr1	25520459	+	CCC	2	10
chr1	25520460	+	CCT	2	10
chr1	25520461	+	CTG	0	10
chr1	25520463	-	CAG	11	11
chr1	25520464	-	CCA	11	11
chr1	25520465	+	CGT	7	9
chr1	25520466	-	CGC	7	7
chr1	25520469	+	CCC	1	10
chr1	25520470	+	CCC	0	10
chr1	25520471	+	CCT	0	10
chr1	25520472	+	CTG	1	10
chr1	25520474	-	CAG	13	13
chr1	25520477	+	CCG	0	9
chr1	25520478	+	CGG	5	10
chr1	25520479	-	CGG	13	13
chr1	25520480	-	CCG	13	13
chr1	25520481	-	CCC	12	13
chr1	25520482	-	CCC	12	12


In [27]:
! for file in `ls output/cz`; do echo ${file} && czip query -I output/cz/${file} -r output/mm10_with_chrL.allc.cz -K chr1 -s 25520457 -e 25520482; done;

FC_E17b_3C_5-5-I24-A21.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	3	3
chr1	25520458	-	CCA	3	3
chr1	25520459	+	CCC	2	2
chr1	25520460	+	CCT	2	2
chr1	25520461	+	CTG	0	2
chr1	25520463	-	CAG	3	3
chr1	25520464	-	CCA	3	3
chr1	25520465	+	CGT	1	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	1	2
chr1	25520470	+	CCC	0	2
chr1	25520471	+	CCT	0	2
chr1	25520472	+	CTG	1	2
chr1	25520474	-	CAG	3	3
chr1	25520477	+	CCG	0	2
chr1	25520478	+	CGG	1	2
chr1	25520479	-	CGG	3	3
chr1	25520480	-	CCG	3	3
chr1	25520481	-	CCC	3	3
chr1	25520482	-	CCC	3	3
FC_M_E15a_3C_1-1-I5-B1.cz
chrom	pos	strand	context	mc	cov
chr1	25520457	-	CAA	1	1
chr1	25520458	-	CCA	1	1
chr1	25520459	+	CCC	0	1
chr1	25520460	+	CCT	0	1
chr1	25520461	+	CTG	0	1
chr1	25520463	-	CAG	1	1
chr1	25520464	-	CCA	1	1
chr1	25520465	+	CGT	0	1
chr1	25520466	-	CGC	1	1
chr1	25520469	+	CCC	0	1
chr1	25520470	+	CCC	0	1
chr1	25520471	+	CCT	0	1
chr1	25520472	+	CTG	0	1
chr1	25520474	-	CAG	2	2
chr1	25520477	+	CCG	0	0
chr1	25520478	+	CGG	0	1
chr1	25520479	-	CGG	

## 5. Build a cell × gene `AnnData`

`cytozip.features.cz_to_anndata` aggregates per-cell `.cz` files over
a feature interval set. When `features=` is a GTF path, cytozip
extracts one interval per gene, merges GENCODE records sharing a
symbol, and extends each interval by `flank_bp` (default 2 kb) on
both sides.


In [28]:
! time czip cz_to_anndata -I output/cz \
    -f /home/x-wding2/Ref/mm10/annotations/gencode.vM23.annotation.gtf \
    -O output/allcells.h5ad -r output/mm10_with_chrL.allc.cz -j 10


real	0m56.459s
user	3m6.523s
sys	2m6.136s


In [29]:
import anndata
adata=anndata.read_h5ad("output/allcells.h5ad")
adata

AnnData object with n_obs × n_vars = 9 × 55228
    obs: 'alpha', 'beta', 'prior_mean'
    var: 'chrom', 'start', 'end', 'gene_id', 'gene_name', 'gene_type', 'strand'
    uns: 'cytozip_score'
    layers: 'cov', 'mc'

In [30]:
adata.obs

,alpha,beta,prior_mean
FC_E17b_3C_5-5-I24-A21,16.552773,4.298903,0.793834
FC_M_E15a_3C_1-1-I5-B1,2.186242,2.139226,0.505435
FC_M_P12b_3C_2-5-M17-N10,2.995694,2.875796,0.510210
FC_M_P3b_3C_6-6-J3-P24,1.944698,1.912869,0.504125
FC_M_P6a_3C_7-3-K21-P5,1.220260,1.223016,0.499436
FC_M_P9B_3C_6-2-F6-O4,0.855686,0.833245,0.506644
FC_P0b_3C_5-1-I24-J14,5.303945,5.269817,0.501614
FC_P13a_3C_7-1-A11-O1,2.719091,2.712517,0.500605
FC_P28a_3C_2-1-E5-N14,2.280605,2.223951,0.506289


In [31]:
adata.var

,chrom,start,end,gene_id,gene_name,gene_type,strand
name,,,,,,,
0610005C13Rik,chr7,45565793,45577327,ENSMUSG00000109644.1,0610005C13Rik,lncRNA,-
0610006L08Rik,chr7,74816817,74855813,ENSMUSG00000108652.1,0610006L08Rik,lncRNA,-
0610009B22Rik,chr11,51683385,51690874,ENSMUSG00000007777.9,0610009B22Rik,protein_coding,-
0610009E02Rik,chr2,26443695,26461390,ENSMUSG00000086714.1,0610009E02Rik,lncRNA,+
0610009L18Rik,chr11,120346677,120353190,ENSMUSG00000043644.4,0610009L18Rik,lncRNA,+
...,...,...,...,...,...,...,...
n-R5s96,chr8,47717245,47721369,ENSMUSG00000064508.1,n-R5s96,rRNA,+
n-R5s97,chr8,47897181,47901294,ENSMUSG00000064519.1,n-R5s97,rRNA,+
n-R5s98,chr8,54864260,54868379,ENSMUSG00000084498.1,n-R5s98,rRNA,+


In [32]:
adata.to_df()

name,0610005C13Rik,0610006L08Rik,0610009B22Rik,0610009E02Rik,0610009L18Rik,0610010F05Rik,0610010K14Rik,0610012D04Rik,0610012G03Rik,0610025J13Rik,...,n-R5s90,n-R5s92,n-R5s93,n-R5s94,n-R5s95,n-R5s96,n-R5s97,n-R5s98,n-TSaga9,n-TStga1
FC_E17b_3C_5-5-I24-A21,0.825397,0.802066,0.800847,0.786429,0.873874,0.800176,0.764977,0.812500,0.724036,0.768344,...,0.868852,0.830882,0.813953,0.772152,0.674359,0.748447,0.850467,0.894231,0.888889,0.857143
FC_M_E15a_3C_1-1-I5-B1,0.474820,0.510067,0.558333,0.445902,1.000000,0.478599,0.421053,0.430556,0.926829,0.470437,...,0.705882,0.600000,0.000000,0.000000,1.000000,0.000000,0.000000,0.548387,0.309942,0.000000
FC_M_P12b_3C_2-5-M17-N10,0.618785,0.466392,0.565574,0.552511,0.412698,0.495077,0.426396,0.615385,0.014925,0.555556,...,0.000000,0.369231,1.000000,0.107143,0.377358,0.319444,0.000000,0.491228,0.453488,0.000000
FC_M_P3b_3C_6-6-J3-P24,0.722222,0.510549,0.000000,0.491803,0.714286,0.587121,0.000000,0.000000,0.444444,0.403727,...,0.000000,0.000000,0.000000,0.526316,0.661538,0.000000,0.822222,0.000000,0.313433,0.393701
FC_M_P6a_3C_7-3-K21-P5,1.000000,0.610465,0.565217,0.380682,0.066667,0.599265,1.000000,0.000000,0.037037,1.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.850000,0.385714,0.000000,0.000000,0.000000
FC_M_P9B_3C_6-2-F6-O4,0.000000,0.324176,0.000000,0.473118,0.142857,0.596273,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.157895,0.000000,0.000000,1.000000
FC_P0b_3C_5-1-I24-J14,0.406542,0.514163,0.522422,0.612132,0.848740,0.562095,0.652778,0.372263,0.577947,0.560440,...,0.655172,0.517857,0.551020,0.428571,0.290323,0.546392,0.258883,0.453659,0.303279,0.876712
FC_P13a_3C_7-1-A11-O1,0.725564,0.474082,0.757143,0.404545,0.326797,0.522672,0.386293,0.423729,0.252033,0.515306,...,0.000000,0.276786,0.615894,0.174603,0.000000,0.526786,0.258621,0.800000,0.800000,0.425532
FC_P28a_3C_2-1-E5-N14,0.315789,0.401980,0.612500,0.341772,0.304348,0.385257,1.000000,0.000000,0.547170,0.628205,...,0.270270,0.766234,1.000000,0.707317,0.000000,0.000000,1.000000,0.000000,0.000000,0.434343
